# BIDMC respiration: deterministic behavioral handoff

This notebook records a first comparison between a raw-data LLM analysis and a deterministic FeatureGraph representation. FeatureGraph does not recognize respiration, decide that a signal is oscillatory, or decide which peaks are meaningful. A researcher supplies explicit transition rules; FeatureGraph applies them reproducibly, preserves boundary provenance, handles incomplete objects, and summarizes the resulting objects without LLM access.

The current native construction uses a fixed rolling envelope: a 100-sample rolling maximum followed by a 100-sample rolling mean and a 100-sample offline alignment shift. Rising and falling states are derived from the one-sample envelope difference. The envelope determines boundaries; raw respiration is retained for amplitude and accumulation measurements. The same parameters are used for every subject.

In [ ]:
import featuregraph as fg
import pandas as pd

In [ ]:
from experiments.bidmc_llm_capture.native_envelope import (
    ENVELOPE_SIGNAL,
    ENVELOPE_SHIFT,
    ENVELOPE_WINDOW,
    add_respiration_envelope,
)

SUBJECT = 1
SAMPLING_RATE = 125

bidmc = fg.datasets.bidmc(subject=SUBJECT)
bidmc = add_respiration_envelope(bidmc)
bidmc = bidmc.dropna(subset=[ENVELOPE_SIGNAL]).copy()

bidmc_oscillation = fg.oscillation.Oscillation(
    signals=ENVELOPE_SIGNAL,
    group="subject",
    smooth_signal=False,
    diff_lag=1,
    eps=0.0,
    max_state_gap=0,
)
bidmc_oscillation_features = bidmc_oscillation.fit_transform(bidmc)
bidmc_oscillation_features["index"] = bidmc_oscillation_features.index

bidmc_oscillation_objects = bidmc_oscillation.summarize(
    bidmc_oscillation_features,
    signal=ENVELOPE_SIGNAL,
    include_partial=True,
)
objects = bidmc_oscillation_objects.table.copy()
object_id = f"{ENVELOPE_SIGNAL}_wave_id"
raw_ranges = (
    bidmc_oscillation_features
    .groupby(["subject", object_id])["respiration_raw"]
    .agg(raw_minimum="min", raw_maximum="max")
    .reset_index()
    .rename(columns={object_id: "oscillation_id"})
)
objects = objects.merge(
    raw_ranges,
    on=["subject", "oscillation_id"],
    validate="one_to_one",
)
objects["envelope_amplitude"] = objects["amplitude"]
objects["amplitude"] = (
    objects["raw_maximum"] - objects["raw_minimum"]
) / 2
complete = objects.loc[objects["is_complete"]].copy()

In [ ]:
assert complete["start_index"].lt(complete["peak_index"]).all()
assert complete["peak_index"].lt(complete["end_index"]).all()
assert complete["temporal_symmetry"].between(0, 1).all()

# Project envelope boundaries onto the raw signal.
for suffix in ("wave_id", "peak", "trough", "wave_complete"):
    bidmc_oscillation_features[f"respiration_{suffix}"] = (
        bidmc_oscillation_features[f"{ENVELOPE_SIGNAL}_{suffix}"]
    )

bidmc_accumulation = fg.accumulation.Accumulation(
    signals="respiration",
    group="subject",
)
bidmc_accumulation_features = bidmc_accumulation.fit_transform(
    bidmc_oscillation_features
)
bidmc_accumulation_objects = bidmc_accumulation.summarize(
    bidmc_accumulation_features,
    signal="respiration",
)

assert len(bidmc_accumulation_objects.table) == len(complete)

In [ ]:
fg.plot(
    bidmc_oscillation_features[6900:8500],
    [
        ["respiration_raw"],
        [ENVELOPE_SIGNAL],
        [f"{ENVELOPE_SIGNAL}_rising", f"{ENVELOPE_SIGNAL}_falling"],
        ["respiration_peak", "respiration_trough"],
        ["respiration_wave_id"],
    ],
)

In [ ]:
def summarize_respiration_objects(respiration_objects, sampling_rate=125):
    complete_objects = respiration_objects.loc[
        respiration_objects["is_complete"]
    ]
    coverage_samples = (
        respiration_objects["end_index"].max()
        - respiration_objects["start_index"].min()
        + 1
    )
    coverage_seconds = coverage_samples / sampling_rate

    return pd.Series(
        {
            "candidate_peaks": int(
                bidmc_oscillation_features[f"{ENVELOPE_SIGNAL}_peak"].sum()
            ),
            "complete_objects": len(complete_objects),
            "partial_objects": (
                len(respiration_objects) - len(complete_objects)
            ),
            "coverage_seconds": coverage_seconds,
            "objects_per_minute": (
                len(complete_objects) / (coverage_seconds / 60)
            ),
            "mean_period_seconds": (
                complete_objects["period"].mean() / sampling_rate
            ),
            "mean_radius_amplitude": (
                complete_objects["amplitude"].mean()
            ),
            "mean_full_excursion": (
                2 * complete_objects["amplitude"].mean()
            ),
            "mean_temporal_symmetry": (
                complete_objects["temporal_symmetry"].mean()
            ),
        }
    )

native_summary = summarize_respiration_objects(
    objects,
    sampling_rate=SAMPLING_RATE,
)
native_summary

## Measurement contracts

- A complete object is a trough–peak–trough sequence with strictly ordered sample indices. Endpoint fragments are retained as partial objects but excluded from complete-object statistics.
- Coverage is the inclusive sample span represented by all returned objects. Rate is complete objects divided by that coverage.
- Period is the distance between consecutive peak indices.
- Envelope states determine boundaries, but FeatureGraph amplitude is half of the raw within-object maximum-minus-minimum range (a radius); full excursion is twice that value.
- Temporal symmetry is `1 - abs(rise_duration - fall_duration) / duration`, bounded from 0 to 1.

These contracts explain why some aggregate values below are comparable and others are not.

In [ ]:
comparison = pd.DataFrame(
    {
        "rolling-envelope FeatureGraph": [
            native_summary["coverage_seconds"],
            native_summary["complete_objects"],
            native_summary["objects_per_minute"],
            native_summary["mean_period_seconds"],
            native_summary["mean_radius_amplitude"],
            native_summary["mean_full_excursion"],
            native_summary["mean_temporal_symmetry"],
        ],
        "prior SciPy-boundary FeatureGraph": [
            479.1, 170, 21.3, 2.82, 0.452, 0.904, -0.550
        ],
        "raw-data LLM run": [
            480.0, 169, 21.1, 2.818, pd.NA, 0.906, 0.857
        ],
        "interpretation": [
            "coverage definitions differ",
            "exact complete-object count agreement with the LLM run",
            "close aggregate agreement",
            "close aggregate agreement",
            "LLM reported full excursion, not radius",
            "comparable after converting FeatureGraph radius",
            "same bounded contract; boundary semantics still differ",
        ],
    },
    index=[
        "coverage_seconds",
        "complete_objects",
        "objects_per_minute",
        "mean_period_seconds",
        "mean_radius_amplitude",
        "mean_full_excursion",
        "mean_temporal_symmetry",
    ],
)
comparison

## Boundary-level validation against BIDMC annotations

The dataset's two human breath-annotation columns provide a check on the native detector. This is not the still-pending FeatureGraph-to-LLM object-level comparison. Each detected peak and annotation is used at most once, and a match must fall within 0.5 seconds (63 samples).

In [ ]:
def match_nearest(detected, reference, tolerance):
    candidates = sorted(
        (abs(left - right), left, right)
        for left in detected
        for right in reference
        if abs(left - right) <= tolerance
    )
    used_detected = set()
    used_reference = set()
    matches = []

    for distance, left, right in candidates:
        if left not in used_detected and right not in used_reference:
            matches.append((left, right, distance))
            used_detected.add(left)
            used_reference.add(right)

    return matches, used_detected, used_reference

annotations = fg.datasets.bidmc_breaths(SUBJECT)
detected_peaks = bidmc_oscillation_features.index[
    bidmc_oscillation_features[f"{ENVELOPE_SIGNAL}_peak"]
].tolist()
tolerance_samples = round(0.5 * SAMPLING_RATE)
annotation_rows = []

for column in [
    "breaths ann1 [signal sample no]",
    "breaths ann2 [signal sample no]",
]:
    reference = annotations[column].dropna().astype(int).tolist()
    matches, used_detected, used_reference = match_nearest(
        detected_peaks,
        reference,
        tolerance_samples,
    )
    distances = pd.Series([match[2] for match in matches])
    annotation_rows.append(
        {
            "annotator": column.split()[1],
            "matched": len(matches),
            "extra_detected": len(detected_peaks) - len(used_detected),
            "missed_reference": len(reference) - len(used_reference),
            "median_abs_error_samples": distances.median(),
            "max_abs_error_samples": distances.max(),
        }
    )

annotation_comparison = pd.DataFrame(annotation_rows)
annotation_comparison

## Accumulation objects

Within each complete oscillation, accumulation is the discrete area of the signal above the object's trough-derived baseline. It describes the magnitude and timing of waveform excursion—including area before and after the peak, centroid time, and half-accumulation time. Because the BIDMC respiration channel is a normalized waveform rather than calibrated airflow or volume, this must not be interpreted as physical inhaled or exhaled volume.

The example below selects the first complete oscillation and the accumulation object carrying the same identifier. The oscillation begins on the trough sample; accumulation begins on the following sample where integration starts, so its start index is one sample later while both end at the same next trough.

In [ ]:
DEMO_OBJECT_ID = int(complete.iloc[0]["oscillation_id"])
oscillation_demo = complete.loc[
    complete["oscillation_id"] == DEMO_OBJECT_ID
].iloc[0]
accumulation_demo = bidmc_accumulation_objects.table.loc[
    bidmc_accumulation_objects.table["accumulation_id"]
    == DEMO_OBJECT_ID
].iloc[0]

object_comparison = pd.DataFrame(
    [
        {
            "representation": "oscillation",
            "object_id": DEMO_OBJECT_ID,
            "boundary summary": (
                f"{oscillation_demo['start_index']:.0f}–"
                f"{oscillation_demo['peak_index']:.0f}–"
                f"{oscillation_demo['end_index']:.0f} "
                "(trough–peak–trough)"
            ),
            "magnitude summary": (
                f"radius amplitude = {oscillation_demo['amplitude']:.3f}"
            ),
            "timing summary": (
                f"temporal symmetry = "
                f"{oscillation_demo['temporal_symmetry']:.3f}"
            ),
            "adds": "boundary geometry and excursion",
        },
        {
            "representation": "accumulation",
            "object_id": DEMO_OBJECT_ID,
            "boundary summary": (
                f"{accumulation_demo['start_index']:.0f}–"
                f"{accumulation_demo['end_index']:.0f} "
                "(integration span)"
            ),
            "magnitude summary": (
                f"total AUC = {accumulation_demo['total_auc']:.3f} "
                "sample-units"
            ),
            "timing summary": (
                f"area symmetry = "
                f"{accumulation_demo['accumulation_symmetry']:.3f}; "
                f"centroid = {accumulation_demo['centroid_time']:.1f} "
                "samples after start"
            ),
            "adds": "baseline-relative area and its timing",
        },
    ]
)

print(
    f"Object {DEMO_OBJECT_ID} is a {oscillation_demo['duration'] / SAMPLING_RATE:.3f}-second "
    f"trough–peak–trough transition with radius amplitude "
    f"{oscillation_demo['amplitude']:.3f}. Its corresponding accumulation "
    f"object contains {accumulation_demo['total_auc']:.3f} normalized "
    f"sample-units of baseline-relative area; "
    f"{accumulation_demo['accumulation_symmetry']:.3f} indicates that this "
    f"area is nearly balanced before and after the peak."
)
object_comparison

## Completed blinded object-level comparison

A context-isolated LLM received only the frozen subject 1 waveform and measurement contract. It selected a fourth-order 0.8 Hz Butterworth filter followed by SciPy `find_peaks` (188-sample minimum distance and 0.08 prominence) on the filtered signal and its negation. It returned 169 complete objects and one trailing partial object without seeing FeatureGraph boundaries, parameters, counts, or prior results.

One-to-one matching within 63 samples pairs all 169 complete LLM objects with all 169 rolling-envelope FeatureGraph objects. There are no unmatched complete objects on either side. Median absolute peak error is 17 samples; both BIDMC annotation series match all 170 detected peaks and miss none.

For matched objects, mean period is 2.821 seconds for both FeatureGraph and the LLM. Mean raw full excursion is 0.903 for both, with median absolute error 0.00489. Mean temporal symmetry is 0.910 for FeatureGraph and 0.844 for the LLM, with median absolute error 0.072. Median trough-boundary error falls from 53 to 33 samples relative to the previous native difference rule.

The LLM's complete method and object table are preserved under `experiments/bidmc_llm_capture/results/`, and the documented SciPy pipeline reproduces all 170 returned rows without further LLM access.

## Rolling-envelope multi-subject transfer result

The fixed 100/100 envelope was applied unchanged to all 52 remaining BIDMC subjects. Across all 53 records, FeatureGraph produces 8,180 complete objects, the recorded LLM-selected baseline produces 7,168, and 6,513 match. FeatureGraph produces 1,667 unmatched objects and misses 655 baseline objects. Median subject-level matched fractions are 88.8% of FeatureGraph objects and 100.0% of baseline objects.

Relative to the prior native difference rule, the envelope adds 313 matches, removes 1,093 FeatureGraph-only objects, misses 313 fewer baseline objects, and reduces median symmetry error from 0.352 to 0.094. Median peak error increases from 12 to 16 samples. Subjects 5, 35, 38, and 39 remain severe failures, so the construction still does not support a universal detector-equivalence claim. Full tables and interpretation are stored in `experiments/bidmc_llm_capture/ENVELOPE_RESULTS.md`.